In [ ]:
import contimod as cm
import contimod_graphene as cmg

import matplotlib.pyplot as plt
import jax
import jax.scipy as jsp

params = cmg.GrapheneTBParameters.preset("tlg").replace(U=62.0)
model = cmg.RhombohedralMultilayer(n_layers=3, params=params)

h_func = model.hamiltonian
h_func2 = lambda kx, ky: jsp.linalg.block_diag(
    h_func(kx, ky),
    h_func(-kx, -ky).conj(),
)

kMesh = cm.hamiltonian_meshgrid.kMeshgrid2D(0.28, 251)
hs = jax.vmap(jax.vmap(h_func2))(*kMesh.ks.T)

h0 = cm.hamiltonian_meshgrid.DiscretizedHamiltonianMeshgrid(
    kMesh, hs, projected_dims=4, T=0.1, chemicalpotential=-31
)
h0.plot_bands_fermisurface(ywindow=(-10, 10), cmap="viridis", figsize=(5, 2))
plt.show()


In [ ]:
import numpy as np

susceptibility_ref_static = h0.get_chargesusceptibility(1e-9, broadening=0.005) #broadening=0.01)
susceptibility_ref_static.data = -susceptibility_ref_static.data

omegas = np.linspace(1e-4, 50, 80)
susceptibility_ref = h0.get_chargesusceptibility(omegas, broadening=0.33) #broadening=0.01)
susceptibility_ref.data = -susceptibility_ref.data


In [ ]:
def plot_maps_with_marginal(da_chi0, figsize=(8, 3)):

    # Create a figure with 1 row and 4 columns
    fig, axs = plt.subplots(1, 4, figsize=figsize, layout="constrained",
                            width_ratios=[1, 0.65, 1, 0.65], height_ratios=[1])

    # -------------------------------
    # (a) Static Data Group
    # Panel 1: 2D map of Re[χ₀(qₓ,q_y,ω=0)] with colorbar on top
    da_chi0.sel(omega=1e-9, method='nearest', tolerance=1e-4).real.plot(
        ax=axs[0],
        x="qx", y="qy",
        cmap="magma",
        center=False,
        add_colorbar=True,
        cbar_kwargs={
            'label': r'$-\mathrm{Re}\chi_0(\omega=0)$  (meV)',
            'orientation': 'horizontal',
            'location': 'top',
            'pad': 0.02
        },
        rasterized=True
    )
    axs[0].set_aspect("equal")
    axs[0].axvline(0, color="black", linestyle=":")
    axs[0].set_xlabel(r'$q_x$  ($a_G$)')
    axs[0].set_ylabel(r'$q_y$  ($a_G$)')
    axs[0].set_title('')

    # Panel 2: Marginal plot (y–cut at qx=0) from static data
    line_odd = da_chi0.sel(omega=0.0, qx=0, method='nearest').real
    # line_even = da_chi0_2.sel(omega=0.0, qx=0, method='nearest').real
    axs[1].plot(line_odd.values, line_odd.qy, '-', label="odd")
    # axs[1].plot(line_even.values, line_even.qy, '--',
    #     # color="gray",
    #     alpha=0.5, label="even")
    axs[1].set_xlabel(r'$-\mathrm{Re}\,\chi_0$')
    plt.setp(axs[1].get_yticklabels(), visible=False)
    # axs[1].legend(handlelength=0.5, ncol=2, loc='upper left', 
    #             labelspacing=0.5, handletextpad=0.5, columnspacing=1.0)
    axs[1].sharey(axs[0])

    # -------------------------------
    # (b) Dynamic Data Group
    # Panel 3: 2D map of Im[χ₀(qₓ,q_y=0,ω)] for qₓ ≥ 0 with colorbar on top
    da_chi0.sel(qy=0, method='nearest').where(da_chi0.qx >= 0, drop=True).imag.plot(
        ax=axs[2],
        x="qx", y="omega",
        cmap="magma",
        # center=0,
        robust=True,
        add_colorbar=True,
        cbar_kwargs={
            'label': r'$-\mathrm{Im}\chi_0$  (meV)',
            'orientation': 'horizontal',
            'location': 'top',
            'pad': 0.02
        },
        rasterized=True
    )
    qxcut = 0.03
    axs[2].set_aspect("auto")
    axs[2].axvline(qxcut, color="lightgray", linestyle=":")
    axs[2].set_xlabel(r'$q_x$  ($a_G$)')
    axs[2].set_ylabel(r'$\omega$ (meV)')
    axs[2].set_title('')

    # Panel 4: Marginal plot (cut at qx=0) from dynamic data
    da_chi0.sel(qx=qxcut, qy=0, method='nearest').imag.plot(
        y="omega",
        ax=axs[3],
        linestyle='-',
        label="odd"
    )
    # da_chi0_2.sel(qx=qxcut, qy=0, method='nearest').imag.plot(
    #     y="omega",
    #     ax=axs[3],
    #     # linestyle='--',
    #     # color="gray",
    #     alpha=0.5,
    #     label="even"
    # )
    axs[3].set_title('')
    axs[3].set_xlabel(r'$-\mathrm{Im}\,\chi_0$')
    axs[3].set_ylabel(None)
    plt.setp(axs[3].get_yticklabels(), visible=False)
    # axs[3].legend(handlelength=0.5, ncol=2, loc='upper left',
    #             labelspacing=0.5, handletextpad=0.5, columnspacing=1.0)
    axs[3].sharey(axs[2])
    # axs[3].axhline(0, color="black", linestyle=":")
    # axs[3].set_ylim(0,5)
    # axs[3].set_xlim(0, 4.5)

    # -------------------------------
    # Add group labels using fig.text so they don't affect constrained_layout
    # Get the axes positions in figure coordinates:
    pos_static = axs[0].get_position()
    pos_dynamic = axs[2].get_position()

    fig.text(pos_static.x0 - 0.08, pos_static.y1 + 0.12, '(a)', 
            fontsize=10, ha='right', va='bottom')
    fig.text(pos_dynamic.x0 + 0.02, pos_dynamic.y1 + 0.12, '(b)', 
            fontsize=10, ha='right', va='bottom')
    
    return fig, axs



def plot_maps2_with_marginal(da_chi0_2, figsize=(8, 3)):

    # Create a figure with 1 row and 4 columns
    fig, axs = plt.subplots(1, 4, figsize=figsize, layout="constrained",
                            width_ratios=[1, 0.65, 1, 0.65], height_ratios=[1])
    
    # fig.set_constrained_layout_pads(w_pad=0.5, h_pad=1.0, hspace=0.2, wspace=0.3)

    # -------------------------------
    # (a) Static Data Group
    # Panel 1: 2D map of Re[χ₀(qₓ,q_y,ω=0)] with colorbar on top
    da_chi0_2.sel(qy=0, method='nearest').where(da_chi0_2.qx >= 0, drop=True).real.plot(
        ax=axs[0],
        x="qx", y="omega",
        cmap="RdBu",
        center=0,
        add_colorbar=True,
        robust=True,
        cbar_kwargs={
            'label': r'$-\mathrm{Re}\chi_0$  (meV)',
            'orientation': 'horizontal',
            'location': 'top',
            'pad': 0.02
        },
        rasterized=True
    )
    axs[0].set_aspect("auto")
    qxcut = 0.03
    axs[0].axvline(qxcut, color="black", linestyle=":")
    axs[0].set_xlabel(r'$q_x$  ($a_G$)')
    axs[0].set_ylabel(r'$\omega$ (meV)')
    axs[0].set_title('')

    # Panel 2: Marginal plot (y–cut at qx=0) from static data
    da_chi0_2.sel(qy=0, qx=qxcut, method='nearest').real.plot(
        y="omega",
        ax=axs[1],
        linestyle='-'
    )
    # line_odd = da_chi0_2.sel(omega=0.0, qx=0, method='nearest').real
    # axs[1].plot(line_odd.values, line_odd.qy, '-')
    axs[1].set_xlabel(r'$-\mathrm{Re}\,\chi_0$')
    plt.setp(axs[1].get_yticklabels(), visible=False)
    axs[1].set_title('')
    axs[1].set_ylabel(None)
    axs[1].axvline(0, color="black", linestyle="-", linewidth=0.5)
    # axs[1].legend(handlelength=0.5, ncol=2, loc='upper left', labelspacing=0.5, handletextpad=0.5, columnspacing=1.0)
    axs[1].sharey(axs[0])

    # -------------------------------
    # (b) Dynamic Data Group
    # Panel 3: 2D map of Im[χ₀(qₓ,q_y=0,ω)] for qₓ ≥ 0 with colorbar on top
    da_chi0_2.sel(qy=0, method='nearest').where(da_chi0_2.qx >= 0, drop=True).imag.plot(
        ax=axs[2],
        x="qx", y="omega",
        # cmap="RdBu",
        # center=0,
        cmap="magma",
        robust=True,
        add_colorbar=True,
        cbar_kwargs={
            'label': r'$-\mathrm{Im}\chi_0$  (meV)',
            'orientation': 'horizontal',
            'location': 'top',
            'pad': 0.02
        },
        rasterized=True
    )
    qxcut = 0.03
    axs[2].set_aspect("auto")
    axs[2].axvline(qxcut, color="lightgray", linestyle=":")
    axs[2].set_xlabel(r'$q_x$  ($a_G$)')
    # axs[2].set_ylabel(r'$\omega$ (meV)')
    axs[2].set_ylabel(None)
    plt.setp(axs[2].get_yticklabels(), visible=False)
    axs[2].set_title('')
    axs[2].sharey(axs[1])

    # Panel 4: Marginal plot (cut at qx=0) from dynamic data
    da_chi0_2.sel(qx=qxcut, qy=0, method='nearest').imag.plot(
        y="omega",
        ax=axs[3],
        linestyle='-'
    )
    axs[3].set_title('')
    axs[3].set_xlabel(r'$-\mathrm{Im}\,\chi_0$')
    axs[3].set_ylabel(None)
    plt.setp(axs[3].get_yticklabels(), visible=False)
    # axs[3].legend(handlelength=0.5, ncol=2, loc='upper left',labelspacing=0.5, handletextpad=0.5, columnspacing=1.0)
    axs[3].sharey(axs[2])
    # axs[3].axhline(0, color="black", linestyle=":")
    # axs[3].set_ylim(0,5)
    # axs[3].set_xlim(0, 4.5)

    # -------------------------------
    # Add group labels using fig.text so they don't affect constrained_layout
    # Get the axes positions in figure coordinates:
    pos_static = axs[0].get_position()
    pos_dynamic = axs[2].get_position()

    fig.text(pos_static.x0 - 0.08, pos_static.y1 + 0.12, '(a)', 
            fontsize=14, fontweight='bold', ha='right', va='bottom')
    fig.text(pos_dynamic.x0 + 0.02, pos_dynamic.y1 + 0.12, '(b)', 
            fontsize=14, fontweight='bold', ha='right', va='bottom')
    
    return fig, axs


In [ ]:
import xarray as xr

chi_ref_da = susceptibility_ref.to_xarray()
chi_static_ref_da = susceptibility_ref_static.to_xarray()
chi_ref_da[0,:,:] = chi_static_ref_da[0,:,:]

fig, axs = plot_maps2_with_marginal(chi_ref_da, figsize=(8, 3))
axs[0].set_xlim(0, 0.14)
axs[2].set_xlim(0, 0.14)
plt.show()

fig, axs = plot_maps_with_marginal(chi_ref_da, figsize=(8, 3))
axs[0].set_xlim(-0.14, 0.14)
axs[0].set_ylim(-0.14, 0.14)
axs[2].set_xlim(0, 0.14)
plt.show()

In [ ]:
# import libtetrabz

# B = np.eye(3) * kMesh.kmax
# B[2,2] *= 20

# mu = h0.chemicalpotential

# mid1 = h0.bands.shape[0]//2 + 1
# windowL = mid1 - h0.bands.shape[0]//4
# windowR = mid1 + h0.bands.shape[0]//4
# nq = h0.bands.shape[0]//4

# freqs = np.linspace(0, 50, 80)
# ImChi = np.zeros((nq,freqs.shape[0]))

# for iqx in range(nq):
#     dqx = iqx
#     dqy = 0
#     qmid = h0.kMesh.ks[mid1, mid1, 0]
#     q = h0.kMesh.ks[mid1+dqx, mid1+dqy, 0]

#     bands0 = h0.bands[windowL:windowR, windowL:windowR, None, :]
#     bands1 = h0.bands[windowL+dqx:windowR+dqx, windowL+dqy:windowR+dqy, None, :]
#     eigvecs0 = h0.psi[windowL:windowR, windowL:windowR, :, :]
#     eigvecs1 = h0.psi[windowL+dqx:windowR+dqx, windowL+dqy:windowR+dqy, :, :]

#     matrixelems = np.abs(np.einsum("ijlm,ijln->ijmn", eigvecs0.conj(), eigvecs1)[:,:,None,:,:])**2

#     wght = libtetrabz.fermigr(B,bands0-mu, bands1-mu, freqs)

#     for iw in range(freqs.shape[0]):
#         ImChi[iqx,iw] = np.sum(matrixelems * wght[...,iw])

# # plot heatmap 
# plt.figure()
# plt.imshow(ImChi.T, aspect="auto", origin="lower", extent=(freqs[0], freqs[-1], 0, nq))
# plt.colorbar()
# plt.show()

In [ ]:
import numba as nb
import numpy as np
import scipy as sp

@nb.njit(parallel=True)
def lorentzian_kernel(delta, gamma):
    return (gamma / np.pi) / (delta**2 + gamma**2) # Standard definition: (1/pi) * [gamma / (delta^2 + gamma^2)]

@nb.njit(parallel=True)
def gaussian_kernel(delta, gamma):
    return (1/(gamma*np.sqrt(2*np.pi))) * np.exp(-0.5*(delta/gamma)**2) # Standard definition: (1/(gamma*sqrt(2*pi))) * exp(-0.5*(delta/gamma)^2)

@nb.njit()
def abs2(z):
    return z.real*z.real + z.imag*z.imag

@nb.njit(parallel=True, fastmath=True)
def get_polarizability_im_histogram(
    omega_bin_edges, f_array, E_array, U_array, 
    nw1, nw2, tolerance=1e-12
):
    """
    Computes the charge susceptibility, storing into 'result_sum'.
    """
    N1, N2, Norb, D = U_array.shape
    Nw = omega_bin_edges.shape[0]

    # Bin edges
    omega_bin_centers = 0.5*(omega_bin_edges[:-1] + omega_bin_edges[1:])
    dws = omega_bin_edges[1:] - omega_bin_edges[:-1]
    nbins = len(omega_bin_centers)

    result_sum = np.zeros((N1, N2, Nw), dtype=np.complex128)

    q1min, q1max = -nw1, nw1
    q2min, q2max = -nw2, nw2
    k1min, k1max = nw1, N1 - nw1
    k2min, k2max = nw2, N2 - nw2

    # Parallelize over the full (q1,q2) range.
    # That is safe because each (q1, q2) updates a unique slice
    # result_sum[N1//2+q1, N2//2+q2, :] (no overlap).
    for q1 in nb.prange(q1min, q1max):
        for q2 in range(q2min, q2max):
            
            # Shifted indices in the result array
            res_q1 = N1//2 + q1
            res_q2 = N2//2 + q2

            for k1 in range(k1min, k1max):
                k1pq = k1 + q1
                for k2 in range(k2min, k2max):
                    k2pq = k2 + q2

                    fermi_kpq = f_array[k1pq, k2pq]
                    fermi_k = f_array[k1, k2]
                    energies_kpq = E_array[k1pq, k2pq]
                    energies_k = E_array[k1, k2]
                    u_kpq = U_array[k1pq, k2pq]
                    u_k = U_array[k1, k2]

                    for b1 in range(D):
                        for b2 in range(D):

                            weight12 = (fermi_kpq[b1]-fermi_k[b2])
                            
                            if np.abs(weight12) < tolerance:
                                continue

                            # place w12 into histogram
                            d12 = -(energies_kpq[b1] - energies_k[b2])

                            k = np.searchsorted(omega_bin_edges, d12) - 1
                            if k >= 0 and k < nbins:
                                Fsq = abs2(np.sum(u_kpq[:, b1].conj() * u_k[:, b2])) # Compute squared modulus of the formfactor matrix for this q-index

                                if k > 0:
                                    result_sum[res_q1,res_q2,k] += -weight12 * Fsq * (omega_bin_edges[k+1] - d12) / dws[k]

                                result_sum[res_q1,res_q2,k+1] += -weight12 * Fsq * (d12 - omega_bin_edges[k]) / dws[k]       

    return result_sum

def get_chargesusceptibility(
    f_array, E_array, psi_array, omegas
):
    """
    High-level wrapper:
      - ensures arrays are contiguous,
      - calls numba kernel,
      - applies a mask.
    """
    # Ensure arrays are at least 1D or contiguous
    f_array  = np.ascontiguousarray(f_array)
    E_array  = np.ascontiguousarray(E_array)
    psi_array = np.ascontiguousarray(psi_array)
    omegas   = np.ascontiguousarray(np.atleast_1d(omegas))

    N1, N2, Norb, D = psi_array.shape
    nw1 = N1 // 4
    nw2 = N2 // 4

    # Call the improved numba version
    output = 1j * np.pi * get_polarizability_im_histogram(
        omegas, f_array, E_array, psi_array,
        nw1, nw2
    )

    # Create a mask for the region of interest
    mask = np.zeros_like(output, dtype=bool)
    # Example, you may want to adjust these slices:
    n1 = output.shape[0] // 4
    n2 = output.shape[1] // 4
    mask[n1:3*n1, n2:3*n2, :] = True

    return np.ma.masked_array(output, mask=~mask)

def broaden_gaussian(chi, omega_bin_edges, broadening):
    org_shape = chi.shape
    chi_flat = chi.reshape(-1, org_shape[-1]) # Flatten all q-indices into one axis

    print("convolving histograms with Lorentzian kernel, broadening = ", broadening)
    for iq in range(chi_flat.shape[0]):
        # out = convolve_histogram_direct(Pi0_flat[:,iq], omega_bin_centers, broadening)
        omega_centered = omega_bin_edges - (omega_bin_edges.max() + omega_bin_edges.min()) / 2
        kernel = gaussian_kernel(omega_centered, broadening)
        # kernel = lorentzian_kernel(omega_centered, broadening)
        out = sp.signal.convolve(chi_flat[iq,:], kernel, mode='same')
        chi_flat[iq,:] = out # Normalize by the bin width

    return chi_flat.reshape(org_shape)

def broaden_lorentzian(chi, omega_bin_edges, broadening):
    org_shape = chi.shape
    chi_flat = chi.reshape(-1, org_shape[-1]) # Flatten all q-indices into one axis

    print("convolving histograms with Lorentzian kernel, broadening = ", broadening)
    for iq in range(chi_flat.shape[0]):
        # out = convolve_histogram_direct(Pi0_flat[:,iq], omega_bin_centers, broadening)
        omega_centered = omega_bin_edges - (omega_bin_edges.max() + omega_bin_edges.min()) / 2
        kernel = lorentzian_kernel(omega_centered, broadening)
        out = sp.signal.convolve(chi_flat[iq,:], kernel, mode='same')
        chi_flat[iq,:] = out # Normalize by the bin width

    return chi_flat.reshape(org_shape)

# def get_polarizability_im(omegas, fermi_k, energies_k, psi_k, broadening=None):
#     """
#     High-level function that accepts formfactors with shape:
#       (q1, q2, ..., qk, nLL, nLL)
#     and returns Pi0 with shape:
#       (len(omegas), q1, q2, ..., qk)
#     """
#     # Extract the shape of the q-indices (all dimensions except the last two)
#     q_shape = psi_k.shape[:-2]
#     nB = psi_k.shape[-1]
    
#     formfactors_flat = psi_k.reshape(-1, nB, nB) # Flatten all q-indices into one axis
#     Pi0_flat, omega_bin_centers = get_polarizability_im_histogram(omegas, fermi_k, energies_k, psi_k)
#     # omega_bin_centers = 0.5*(omega_bin_centers[:-1] + omega_bin_centers[1:])

#     if broadening is not None:
#         # Convolve each histogram with the Lorentzian kernel
#         print("convolving histograms with Lorentzian kernel, broadening = ", broadening)
#         for iq in range(Pi0_flat.shape[1]):
#             # out = convolve_histogram_direct(Pi0_flat[:,iq], omega_bin_centers, broadening)
#             omega_centered = omega_bin_centers - (omega_bin_centers.max() + omega_bin_centers.min()) / 2
#             kernel = gaussian_kernel(omega_centered, broadening)
#             # kernel = lorentzian_kernel(omega_centered, broadening)
#             out = sp.signal.convolve(Pi0_flat[:,iq], kernel, mode='same')
#             Pi0_flat[:,iq] = out # Normalize by the bin width
    
#     # set zero frequency im to zero
#     sel = (np.abs(omega_bin_centers) < 1e-9)
#     Pi0_flat[sel, :] = Pi0_flat[sel,:].real 
#     return 1j * np.pi * Pi0_flat.reshape((len(omega_bin_centers),) + q_shape), omega_bin_centers

def get_frequency_grid(h, dw=0.15, verbose=False):
    omega_max = h.bands.max() - h.bands.min()
    omegas  = np.arange(1e-10, omega_max/4, dw)
    omegas2 = np.arange(omega_max/4, omega_max, dw)
    # omegas2 = np.geomspace(omegas2[0], omega_max, 500)
    omegas = np.concatenate((omegas, omegas2))
    if verbose:
        print(f"GRID INFO: omega_max={omega_max:.2f}, n_omega={len(omegas)}, dw={dw:.2f} meV")
    return omegas

In [ ]:
# import xarray as xr

# # omegas = np.linspace(1e-4, 50, 200)
# omegas = get_frequency_grid(h0, dw=0.15, verbose=True)
# broadening = 0.1
# chi_im = get_chargesusceptibility(h0.fermidirac, h0.bands, h0.psi, omegas)
# chi_im = broaden_lorentzian(chi_im, omegas, broadening)

# chi_im_da = xr.DataArray(
#     np.moveaxis(-chi_im, -1, 0),
#     dims=["omega", "qx", "qy"],
#     coords={
#         "qx": h0.kMesh.kx,
#         "qy": h0.kMesh.ky,
#         "omega": omegas
#     },
#     attrs={
#         "description": "Charge susceptibility",
#         "units": "meV",
#         "broadening": broadening
#     }
# )

# fig, axs = plot_maps2_with_marginal(chi_im_da*kMesh.weight*4, figsize=(8, 3))
# # axs[0].set_xlim(-0.14, 0.14)
# # axs[0].set_ylim(-0.14, 0.14)
# axs[0].set_xlim(0, 0.14)
# axs[2].set_xlim(0, 0.14)
# plt.show()

In [ ]:
import xarray as xr

omegas = get_frequency_grid(h0, dw=0.15, verbose=True)
chi_im = get_chargesusceptibility(h0.fermidirac, h0.bands, h0.psi, omegas)
chi_im = np.moveaxis(chi_im, -1, 0)

In [ ]:
broadening = np.zeros(omegas.shape[0], dtype=np.float64)
broadening[0:-1] = (omegas[1:] - omegas[:-1])*2
broadening[-1] = broadening[-2]

chi  = cm.graphene_LL.hilbert_custom(np.array(chi_im.imag), omegas, broadening)
chi = -(chi + chi[:, ::-1, ::-1])

chi_im_da = xr.DataArray(
    chi,
    dims=["omega", "qx", "qy"],
    coords={
        "qx": h0.kMesh.kx,
        "qy": h0.kMesh.ky,
        "omega": omegas
    },
    attrs={
        "description": "Charge susceptibility",
        "units": "meV",
        # "broadening": broadening
    }
)


In [ ]:
fig, axs = plot_maps2_with_marginal(-chi_im_da*kMesh.weight*4, figsize=(8, 3))
axs[0].set_xlim(0, 0.14)
axs[0].set_ylim(0, 50)
axs[2].set_xlim(0, 0.14)
axs[2].set_ylim(0, 50)
plt.show()
